# Multiple linear regression

Original experiment from [Machine_Learning](https://github.com/Muhammad-Huzifa/Machine_Learning/blob/41d456774a4e2c52e7b6d701ea244eec7b903a88/Mutiple_Linear_Regression_from_Scratch.ipynb). Run from a fresh kernel. See [dataset and environment instructions](../../docs/DATASETS.md).

Framework training, downloaded datasets, and custom image inputs require the dependencies and resources described in the collection README. This migration has checked syntax and paths; it has not repeated every training experiment.

In [ ]:
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "requirements/base.txt").is_file())
TABULAR = ROOT / "data/tabular"
IMAGES = ROOT / "data/images"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)


### Multiple Variable Linear Regression

In [ ]:
import copy, math
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
X_train = np.array([[2104, 5, 1, 45], [1416, 3, 2, 40], [852, 2, 1, 35]])
y_train = np.array([460, 232, 178])
''''
Size (sqft)	Number of Bedrooms	Number of floors	Age of Home	Price (1000s dollars)
    2104	         5	               1	              45         	460
    1416	         3	               2 	              40	        232
    852	             2	               1	              35	        178
'''''
X_train.shape, y_train.shape,print(X_train , y_train)

### Intail Value of b and W 

In [ ]:
b_init = 785.1811367994083
w_init = np.array([ 0.39133535, 18.75376741, -53.36032453, -26.42131618])





##### 𝑓𝐰,𝑏(𝐱)=𝑤0𝑥0+𝑤1𝑥1+...+𝑤𝑛−1𝑥𝑛−1+𝑏
#### 𝑓𝐰,𝑏(𝐱)=𝐰⋅𝐱+𝑏

### Single Prediction

In [ ]:
def predict_single(x, w, b): 

       p = np.dot(x,w) + b 
       return p 

x_vec = X_train[0,:]
print(x_vec)
# Prediction is:
f_wb = predict_single(x_vec, w_init, b_init)
print(f_wb)

### Cost Function

##### 𝐽(𝐰,𝑏)=12𝑚∑𝑖=0𝑚−1(𝑓𝐰,𝑏(𝐱(𝑖))−𝑦(𝑖))2 where: 𝑓𝐰,𝑏(𝐱(𝑖))=𝐰⋅𝐱(𝑖)+b

In [ ]:
def compute_cost(X, y, w, b):
    m = X.shape[0]
    cost = 0.0
    for i in range(m):                                         
        cost = cost + (np.dot(X[i], w) + b - y[i])**2   / (2 * m)                    
    return cost
cost = compute_cost(X_train, y_train, w_init, b_init)
print(f'Cost at optimal w : {cost}')

### Gradient_Descent with Multiple Variables

## ∂𝐽(𝐰,𝑏)/∂𝑤𝑗 =1𝑚∑𝑖=0𝑚−1(𝑓𝐰,𝑏(𝐱(𝑖))−𝑦(𝑖))𝑥(𝑖)𝑗
## ∂𝐽(𝐰,𝑏)/∂𝑏 =1𝑚∑𝑖=0𝑚−1(𝑓𝐰,𝑏(𝐱(𝑖))−𝑦(𝑖))

In [ ]:
def compute_gradient(X, y, w, b): 

    m,n = X.shape           
    dj_dw = np.zeros((n,))
    dj_db = 0.

    for i in range(m):                             
        err = (np.dot(X[i], w) + b) - y[i]   
        for j in range(n):                         
            dj_dw[j] = dj_dw[j] + err * X[i, j]    
        dj_db = dj_db + err                        
    dj_dw = dj_dw / m                                
    dj_db = dj_db / m                                

    return dj_db, dj_dw
tmp_dj_db, tmp_dj_dw = compute_gradient(X_train, y_train, w_init, b_init)
print(f'dj_db at initial w,b: {tmp_dj_db}')
print(f'dj_dw at initial w,b: \n {tmp_dj_dw}')

In [ ]:
def gradient_descent(X, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters): 
    J_history = []
    w = copy.deepcopy(w_in)
    b = b_in

    for i in range(num_iters):

        # Calculate the gradient and update the parameters
        dj_db,dj_dw = gradient_function(X, y, w, b)   ##None

        # Update Parameters using w, b, alpha and gradient
        w = w - alpha * dj_dw               ##None
        b = b - alpha * dj_db               ##None

        # Save cost J at each iteration
        if i<100000:      # prevent resource exhaustion 
            J_history.append( cost_function(X, y, w, b))

        # Print cost every at intervals 10 times or as many iterations if < 10
        if i% math.ceil(num_iters / 10) == 0:
            print(f"Iteration {i:4d}: Cost {J_history[-1]:8.2f}   ")

    return w, b, J_history #return final w,b and J history for graphing
# initialize parameters
initial_w = np.zeros_like(w_init)
initial_b = 0.
# some gradient descent settings
iterations = 1000
alpha = 5.0e-7
# run gradient descent 
w_final, b_final, J_hist = gradient_descent(X_train, y_train, initial_w, initial_b,
                                                    compute_cost, compute_gradient, 
                                                    alpha, iterations)
print(f"b,w found by gradient descent: {b_final:0.2f},{w_final} ")
m,_ = X_train.shape
for i in range(m):
    print(f"prediction: {np.dot(X_train[i], w_final) + b_final:0.2f}, target value: {y_train[i]}")

In [ ]:
# plot cost versus iteration  
fig, (ax1, ax2) = plt.subplots(1, 2, constrained_layout=True, figsize=(12, 4))
ax1.plot(J_hist)
ax2.plot(100 + np.arange(len(J_hist[100:])), J_hist[100:])
ax1.set_title("Cost vs. iteration");  ax2.set_title("Cost vs. iteration (tail)")
ax1.set_ylabel('Cost')             ;  ax2.set_ylabel('Cost') 
ax1.set_xlabel('iteration step')   ;  ax2.set_xlabel('iteration step') 
plt.show()